# 朴素贝叶斯-垃圾邮件识别；

本项目使用朴素贝叶斯算法实现垃圾邮件识别器。

In [1]:
import pandas as pd;
import numpy as np;
import re;
from collections import defaultdict;
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix;

In [2]:
# 加载数据
df = pd.read_csv('spam_ham_dataset.csv');

print(f"数据集大小: {len(df)}");
print(f"正常邮件(ham)数量: {sum(df['label_num'] == 0)}");
print(f"垃圾邮件(spam)数量: {sum(df['label_num'] == 1)}");

print(df[:10]);

数据集大小: 5171
正常邮件(ham)数量: 3672
垃圾邮件(spam)数量: 1499
   Unnamed: 0 label                                               text  \
0         605   ham  Subject: enron methanol ; meter # : 988291\r\n...   
1        2349   ham  Subject: hpl nom for january 9 , 2001\r\n( see...   
2        3624   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3        4685  spam  Subject: photoshop , windows , office . cheap ...   
4        2030   ham  Subject: re : indian springs\r\nthis deal is t...   
5        2949   ham  Subject: ehronline web address change\r\nthis ...   
6        2793   ham  Subject: spring savings certificate - take 30 ...   
7        4185  spam  Subject: looking for medication ? we ` re the ...   
8        2641   ham  Subject: noms / actual flow for 2 / 26\r\nwe a...   
9        1870   ham  Subject: nominations for oct . 21 - 23 , 2000\...   

   label_num  
0          0  
1          0  
2          0  
3          1  
4          0  
5          0  
6          0  
7          1  
8

In [3]:
from sklearn.model_selection import train_test_split;
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label_num'], test_size=0.2, random_state=42
);

print(f"训练集大小: {len(X_train)}");
print(f"测试集大小: {len(X_test)}");

训练集大小: 4136
测试集大小: 1035


In [4]:
class NaiveBayesSpamClassifier:
    """朴素贝叶斯垃圾邮件分类器"""
    def __init__(self):
        self.spam_word_counts = defaultdict(int); # 记录每个单词在垃圾邮件中的出现次数。
        self.ham_word_counts = defaultdict(int);  # 记录每个单词在正常邮件中的出现次数。
        self.spam_total_words = 0;                          # 记录垃圾邮件中的总单词数。
        self.ham_total_words = 0;                           # 记录正常邮件中的总单词数。
        self.spam_docs = 0;                                 # 记录垃圾邮件的文档数。
        self.ham_docs = 0;                                  # 记录正常邮件的文档数。
        self.vocab = set();                            # 记录所有单词。
        self.p_spam = 0;                                    # 记录垃圾邮件的先验概率。
        self.p_ham = 0;                                     # 记录正常邮件的先验概率。
    
    def preprocess_text(self, text):
        """文本预处理：转小写、移除非字母字符、分词"""
        text = text.lower();
        text = re.sub(r'[^a-z\s]', ' ', text);
        words = text.split();
        return [word for word in words if len(word) > 2];
    
    def fit(self, X, y):
        """训练模型"""
        for text, label in zip(X, y):
            words = self.preprocess_text(text);
            
            if label == 1:  # 垃圾邮件：
                self.spam_docs += 1;
                for word in words:
                    self.spam_word_counts[word] += 1;
                    self.spam_total_words += 1;
                    self.vocab.add(word);
            else:  # 正常邮件：
                self.ham_docs += 1;
                for word in words:
                    self.ham_word_counts[word] += 1;
                    self.ham_total_words += 1;
                    self.vocab.add(word);
        
        total_docs = self.spam_docs + self.ham_docs;
        self.p_spam = self.spam_docs / total_docs;
        self.p_ham = self.ham_docs / total_docs;
    
    def predict_single(self, text):
        """预测单条文本"""
        words = self.preprocess_text(text);
        vocab_size = len(self.vocab);
        
        # 使用对数概率避免数值下溢
        log_p_spam = np.log(self.p_spam);
        log_p_ham = np.log(self.p_ham);
        
        for word in words:
            # 拉普拉斯平滑
            spam_count = self.spam_word_counts.get(word, 0);
            ham_count = self.ham_word_counts.get(word, 0);
            
            p_word_given_spam = (spam_count + 1) / (self.spam_total_words + vocab_size);
            p_word_given_ham = (ham_count + 1) / (self.ham_total_words + vocab_size);
            
            log_p_spam += np.log(p_word_given_spam);
            log_p_ham += np.log(p_word_given_ham);
        
        return 1 if log_p_spam > log_p_ham else 0;
    
    def predict(self, X): # 这个函数不用细看！
        """
        预测多条文本。
        就是多次调用 predict_single();
        """
        return [self.predict_single(text) for text in X];

## 训练:

In [5]:
classifier = NaiveBayesSpamClassifier(); # 示例化模型。
classifier.fit(X_train, y_train);

## 评估:

In [6]:
y_pred = classifier.predict(X_test); # 

accuracy = accuracy_score(y_test, y_pred);
precision = precision_score(y_test, y_pred);
recall = recall_score(y_test, y_pred);
f1 = f1_score(y_test, y_pred);

print(f"准确率 (Accuracy): {accuracy:.4f}");
print(f"精确率 (Precision): {precision:.4f}");
print(f"召回率 (Recall): {recall:.4f}");
print(f"F1分数 (F1-Score): {f1:.4f}");

准确率 (Accuracy): 0.9807
精确率 (Precision): 0.9627
召回率 (Recall): 0.9693
F1分数 (F1-Score): 0.9660


In [7]:
# 测试示例
test_emails = [
    "Subject: Congratulations! You have won a $1000 gift card. Click here to claim.", # 主题：恭喜！您已赢得1000美元礼品卡。点击此处领取。
    "Subject: Meeting schedule for next week. Please review the attached document.",  # 主题：下周会议安排。请查看附件文件。
    "Subject: Get rich quick! Buy cheap viagra now!!! Limited time offer!",           # 主题：快点致富！现在就买便宜的伟哥!!限时优惠！
    "Subject: Project update - Q3 financial report attached for your review."         # 主题：项目更新 - 附上第三季度财务报告供您审阅。
];

for email in test_emails:
    prediction = classifier.predict_single(email);
    label = "垃圾邮件(spam)" if prediction == 1 else "正常邮件(ham)";
    print(f"\n邮件: {email}");
    print(f"预测结果: {label}");


邮件: Subject: Congratulations! You have won a $1000 gift card. Click here to claim.
预测结果: 垃圾邮件(spam)

邮件: Subject: Meeting schedule for next week. Please review the attached document.
预测结果: 正常邮件(ham)

邮件: Subject: Get rich quick! Buy cheap viagra now!!! Limited time offer!
预测结果: 垃圾邮件(spam)

邮件: Subject: Project update - Q3 financial report attached for your review.
预测结果: 正常邮件(ham)
